In [2]:
import preprocess as pre
from pathlib import Path
import pandas as pd
import importlib.util
import anndata as ad
import scanpy as sc

In [3]:
# Determine parent folder
base_dir = Path.cwd().parent
path = base_dir / "pipeline_paths_mnt.py"

# Import module dynamically
spec = importlib.util.spec_from_file_location("ppaths", path)
ppaths = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ppaths)

In [4]:
# VARIABLES
# Quality control
min_cells = 200 
min_genes = 200
mito_threshold: float = 5.0
min_cells_per_subject: int = 30

# Highly Variable Gene selection
nr_hvgs = 1000

# train-test split
test_size: float = 0.2
random_state: int = 17

# to stratify AD+NCI v MCIP
stratify_key: str = 'label'

In [5]:
# sets variables in file and folder names
run_vars = f'mg_{min_genes}_mc_{min_cells}_mhvg{nr_hvgs}' 

# to keep consistent paths
pp = ppaths.PipelinePaths("dev", full_pipeline_run_vars=run_vars)

base path found (/mnt/12tb_dsk3/workspaces): True
data path found (/mnt/12tb_dsk3/workspaces/isabell): True


In [ ]:
in_path = Path(pp.conv_data_path)

# datasets to include (keys are readable labels)
datasets = {}

COMBINED = True
if COMBINED:
    cell_types = ['Inh', 'Exc']
    combined_file = in_path / f'{'_'.join(cell_types)}_combined.h5ad'

    if combined_file.exists():
        adata_combined = ad.read_h5ad(combined_file)
    else:
        input_files = [c + '_combined.h5ad' for c in cell_types]
        data = [ad.read_h5ad(in_path / f'{i}') for i in input_files]
        adata_combined = ad.concat(data, join='outer')
        adata_combined.write_h5ad(combined_file)
    datasets[f"'_'.join(cell_types)"] = adata_combined
else:
    input = 'Exc_combined'
    datasets[input] = ad.read_h5ad(in_path / f'{input}.h5ad')

In [7]:
# get the order of the original genes (for verification after processing)
genes_ordered = datasets[list(datasets.keys())[0]].var_names.tolist()

In [8]:
# create labels for AD, NCI and MCIP status, 
# based on cogdx, braaksc, and ceradc scores
for label, adata in datasets.items():
    datasets[label] = pre.create_new_labels(adata, drop_unknown=True)

Stratifying subjects based on cogdx, braaksc, and ceradsc...
Dropping subjects with 'unknown' label...
Removed 217 subjects. (427 -> 210)


    Stratification summary:
    AD: 102
    NCI: 72
    MCIP: 36
          


In [9]:
# save genes that exist in reactome to file
reactome_genes = pre.save_reactome_genes(pp.pathway_data_path, 
        filename_read='ReactomePathways.gmt', 
        filename_save='reactome_genes.txt'
        )

Saving list of genes that are in Reactome
11939 Reactome genes written to /mnt/12tb_dsk3/workspaces/isabell/PathwayData/reactome_genes.txt


In [10]:
# to inspect final objects
final_datasets = {}

for label, adata in datasets.items():
    # keep only genes that are in Reactome
    adata = pre.filter_reactome_genes(adata, set(reactome_genes))

    ## Quality Control
    # remove cells with less than n expressed genes
    adata = pre.filter_cells_by_min_genes(adata, min_genes=min_genes)

    # filter genes to keep only genes expressed in at least n cells
    adata = pre.filter_genes_by_min_expression(adata, min_cells=min_cells)

    # remove cells with mito gene expression over threshold
    adata = pre.filter_cells_by_mitochondrial_content(adata, mt_threshold=mito_threshold)

    # filter genes to keep only subjects with at least n cells
    adata = pre.filter_low_cell_subjects(adata, min_cells_per_subject=min_cells_per_subject)

    datasets[label] = adata
    
    # Stratify AD+NCI, MCIP
    adata, mcip_adata = pre.separate_stratified_data(
            adata,
            save_dir=pp.stratified_full_pipe,
            file_prefix=[f"{label}_AD_NCI", f"{label}_MCIP"]
        )

    # Split AD+NCI into train/test
    train_subjects, test_subjects = pre.train_test_split_subjects(
            adata,
            test_size=test_size,
            random_state=random_state,
            stratify_key=stratify_key,
            save_dir=pp.stratified_path,
            file_prefix=label
        )

    # Split the AnnData object into training and testing sets based on the provided subject lists
    train_adata, test_adata = pre.train_test_split_adata(
        adata,
        train_subjects=train_subjects,
        test_subjects=test_subjects,
        save_dir=pp.stratified_path,
        file_prefix=label
    )

    # HVG selection from train set
    hvgs = pre.select_hvgs(train_adata, nr_hvgs)

    # Filter hvgs
    train_adata = pre.filter_hvgs(train_adata, hvgs)
    test_adata = pre.filter_hvgs(test_adata, hvgs)
    mcip_adata = pre.filter_hvgs(mcip_adata, hvgs)

    # pseudobulk
    # sum counts per subject and high res cell type
    train_adata = pre.aggregate_to_pseudobulk(
        train_adata, 
        bulk_by='mean', 
        )
    
    test_adata = pre.aggregate_to_pseudobulk(
        test_adata, 
        bulk_by='mean', 
        )
    
    mcip_adata = pre.aggregate_to_pseudobulk(
        mcip_adata, 
        bulk_by='mean', 
        )

    # standardize train set, apply transformation to test and MCIP sets
    train_adata_standardized, scaler = pre.standardize(train_adata)
    test_adata_standardized = pre.standardize(test_adata, scaler)
    mcip_adata_standardized = pre.standardize(mcip_adata, scaler)

    # Save the standardized training, testing and MCIP sets to disk as pandas DataFrames
    train_adata_standardized.to_df().reset_index().to_csv(
        pp.compl_full_pipe_path / f'{label}_train_preprocessed.csv', 
        index=False)
    
    test_adata_standardized.to_df().reset_index().to_csv(
        pp.compl_full_pipe_path / f'{label}_test_preprocessed.csv', 
        index=False)
    
    mcip_adata_standardized.to_df().reset_index().to_csv(
        pp.compl_full_pipe_path / f'{label}_mcip_preprocessed.csv', 
        index=False)

    # Save the standardized training, testing and MCIP sets to disk as anndata files
    sc.write(pp.compl_full_pipe_path / f'{label}_train_preprocessed.h5ad', 
             train_adata_standardized)
    sc.write(pp.compl_full_pipe_path / f'{label}_test_preprocessed.h5ad', 
             test_adata_standardized)
    sc.write(pp.compl_full_pipe_path / f'{label}_mcip_preprocessed.h5ad', 
             mcip_adata_standardized)

    final_datasets[f'train_{label}'] = train_adata_standardized
    final_datasets[f'test_{label}'] = test_adata_standardized
    final_datasets[f'mcip_{label}'] = mcip_adata_standardized

Filtering AnnData object to only include genes present in the Reactome database...
Removed 8029 genes. (18934 -> 10905)

Filtering cells with <200 genes...
Filtered data, shape=(695074, 10905)
695157 -> 695074 cells (removed 83)

Filtering genes expressed in <200 cells...
Removed 1139 genes. (10905 -> 9766)

Filtering cells with > 5.0% mitochondrial genes...
Removed 236744 cells. (695074 -> 458330)

Filtering subjects with < 30 cells...
Removed 6 subjects. (209 -> 203)

Separating AnnData object into stratified datasets...
AD+NCI dataset: 394944 cells from 167 subjects
MCIP dataset: 63320 cells from 36 subjects

Saved AD+NCI dataset to /mnt/12tb_dsk3/workspaces/isabell/processed_data/stratified/full_pipeline/mg_200_mc_200_mhvg1000/combined_AD_NCI_adata.h5ad
Saved MCIP dataset to /mnt/12tb_dsk3/workspaces/isabell/processed_data/stratified/full_pipeline/mg_200_mc_200_mhvg1000/combined_MCIP_adata.h5ad

Saved AD+NCI metadata to /mnt/12tb_dsk3/workspaces/isabell/processed_data/stratified/fu

/usr/lib/python3.12/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Shape of pseudobulk AnnData: (133, 1000)

Aggregating single-cell data to pseudobulk by mean...
Shape of pseudobulk AnnData: (34, 1000)

Aggregating single-cell data to pseudobulk by mean...
Shape of pseudobulk AnnData: (36, 1000)

Standardizing data (zero mean, unit variance)...
Standardizing test data using training statistics...
Standardizing test data using training statistics...


In [11]:
# verify genes in correct order
verify = {'train': train_adata, 'test': test_adata, 'mcip':mcip_adata}
pre.verify_gene_order(verify, genes_ordered)

SUCCESS: All datasets maintain original relative gene order.


In [12]:
for label, adata in final_datasets.items():
    print(adata)


AnnData object with n_obs × n_vars = 133 × 1000
    obs: 'n_obs_aggregated', 'projid', 'individual_ID', 'cell_type_high_resolution', 'major_cell_type', 'Pathologic_diagnosis_of_AD', 'batch', 'individualID', 'individualIdSource', 'sex', 'Study', 'msex', 'educ', 'spanish', 'apoe_genotype_x', 'age_at_visit_max', 'age_first_ad_dx', 'age_death_x', 'cts_mmse30_first_ad_dx', 'cts_mmse30_lv', 'braaksc', 'ceradsc', 'cogdx', 'dcfdx_lv', 'clinical_diagnosis', 'pathological_diagnosis', 'clinical_pathological_AD', 'clinical_AD_Label', 'Pathological_AD_Label', 'Clinical_Pathological_AD_Label', 'pmi', 'race', 'study', 'scaled_to', 'apoe_genotype_y', 'tomm40_hap', 'synap_6_complex1', 'synap_6_complex2', 'synap_6_snap25', 'synap_6_synaptophys', 'synap_6_syntaxin', 'synap_6_vamp', 'zcapture_syn_6', 'zcomplexin_6', 'zsnare_6', 'cognep_demog_slope', 'cognep_path_slope', 'cogng_demog_slope', 'cogng_path_slope', 'cognpo_demog_slope', 'cognpo_path_slope', 'cognps_demog_slope', 'cognps_path_slope', 'cognse_de